# SMOLVLA：保护训练与评估

本 Notebook 只执行 protected recipe。训练、checkpoint 保存、耗时记录、14 条固定 seed 闭环评估、视频和动作序列图都在 Notebook kernel 内完成。评估严格绑定本次训练生成的 `TRAINED_POLICY_PATH`，不会回退到历史或预训练权重。


In [ ]:
from pathlib import Path
import json
import os
import shlex
import shutil
import subprocess
import sys

try:
    from IPython.display import HTML, Markdown, display
except Exception:
    class Markdown(str):
        pass

    class HTML(str):
        pass

    def display(obj):
        print(obj)


def find_topic_root():
    cwd = Path.cwd().resolve()
    for candidate in [cwd, *cwd.parents]:
        if (candidate / "assets" / "metrics_snapshot.json").exists():
            return candidate
    raise RuntimeError("请从 AMD ROCm 专题目录或 notebooks 子目录启动 Jupyter。")


TOPIC_ROOT = find_topic_root()
ASSET_DIR = TOPIC_ROOT / "assets"
PROJECT_ROOT = Path(os.environ.get("PROJECT_ROOT", "/path/to/04mujoco复现ACT、Pi0、SmolVLA"))
DATA_ROOT = Path(os.environ.get("DATA_ROOT", "/path/to/datasets/every_embodied"))
MODEL_ROOT = Path(os.environ.get("MODEL_ROOT", "/path/to/model/checkpoints"))
OUTPUT_ROOT = Path(os.environ.get("OUTPUT_ROOT", TOPIC_ROOT / 'outputs' / 'protected' / 'smolvla'))

# The AMD teaching workflow should be runnable from local datasets/checkpoints.
# Avoid surprising network calls during class or when AUP/Radeon Cloud cannot
# reach Hugging Face.
os.environ.setdefault("HF_HUB_OFFLINE", os.environ.get("NOTEBOOK_HF_OFFLINE", "1"))
os.environ.setdefault("TRANSFORMERS_OFFLINE", os.environ.get("NOTEBOOK_HF_OFFLINE", "1"))
os.environ.setdefault("HF_DATASETS_OFFLINE", os.environ.get("NOTEBOOK_HF_OFFLINE", "1"))
os.environ.setdefault("HF_HOME", str(Path(os.environ.get("CACHE_ROOT", OUTPUT_ROOT / "cache")) / "huggingface"))
os.environ.setdefault("HF_DATASETS_CACHE", str(Path(os.environ["HF_HOME"]) / "datasets"))

def public_path(path):
    path = Path(path)
    replacements = [
        (TOPIC_ROOT, "$TOPIC_ROOT"),
        (PROJECT_ROOT, "$PROJECT_ROOT"),
        (DATA_ROOT, "$DATA_ROOT"),
        (MODEL_ROOT, "$MODEL_ROOT"),
        (OUTPUT_ROOT, "$OUTPUT_ROOT"),
    ]
    value = str(path)
    for root, label in sorted(replacements, key=lambda item: len(str(item[0])), reverse=True):
        root_value = str(root)
        if root_value and value.startswith(root_value):
            return label + value[len(root_value):]
    return value


print("TOPIC_ROOT = $TOPIC_ROOT")
print("PROJECT_ROOT =", public_path(PROJECT_ROOT))
print("DATA_ROOT =", public_path(DATA_ROOT))
print("MODEL_ROOT =", public_path(MODEL_ROOT))
print("OUTPUT_ROOT =", public_path(OUTPUT_ROOT))


In [ ]:
def md_table(headers, rows):
    lines = ["| " + " | ".join(headers) + " |", "| " + " | ".join(["---"] * len(headers)) + " |"]
    for row in rows:
        lines.append("| " + " | ".join(public_path(x) if isinstance(x, (str, Path)) else str(x) for x in row) + " |")
    display(Markdown("\n".join(lines)))


def show_json(path, max_chars=5000):
    path = Path(path)
    if not path.exists():
        print("文件不存在：", public_path(path))
        return None
    data = json.loads(path.read_text(encoding="utf-8"))
    text = json.dumps(data, ensure_ascii=False, indent=2)
    print(text[:max_chars] + ("\n..." if len(text) > max_chars else ""))
    return data


def show_video(filename, title):
    path = ASSET_DIR / filename
    display(Markdown(f"**{title}**"))
    if path.exists():
        rel = f"../assets/{filename}"
        display(HTML(f"<video controls muted preload='metadata' width='960'><source src='{rel}' type='video/mp4'></video>"))
    else:
        print("缺少视频素材：", public_path(path))


def show_image(filename, title, width=960):
    path = ASSET_DIR / filename
    display(Markdown(f"**{title}**"))
    if path.exists():
        rel = f"../assets/{filename}"
        display(HTML(f"<img src='{rel}' width='{width}'>"))
    else:
        print("缺少图片素材：", public_path(path))


def run_cmd_preview(command, cwd=None):
    shown = [public_path(x) if isinstance(x, (str, Path)) else x for x in command]
    print("$", shlex.join([str(x) for x in shown]))
    if cwd:
        print("cwd =", public_path(cwd))


def tail_log(log_path, lines=40):
    path = Path(log_path)
    if not path.exists():
        print("日志不存在：", public_path(path))
        return
    content = path.read_text(encoding="utf-8", errors="replace").splitlines()
    print("\n".join(content[-lines:]))


def env_flag(name, default=False):
    value = os.environ.get(name)
    if value is None:
        return default
    return value.strip().lower() in {"1", "true", "yes", "y", "on"}


RUN_SMOKE = env_flag("RUN_SMOKE")
RUN_LONG_TRAIN = env_flag("RUN_LONG_TRAIN", True)
RUN_EVAL = env_flag("RUN_EVAL")
EVAL_SCRIPT = Path(os.environ.get("EVAL_SCRIPT", PROJECT_ROOT / "eval_policy_success.py"))


_XVFB_PROCESS = None


def ensure_xvfb_display():
    """Start a lightweight virtual display for headless MuJoCo evaluation."""
    global _XVFB_PROCESS
    if os.environ.get("DISPLAY"):
        print("DISPLAY =", os.environ["DISPLAY"])
        return None
    xvfb_bin = shutil.which("Xvfb")
    if not xvfb_bin:
        print("没有发现 Xvfb；如遇 GLFW DISPLAY 报错，请先安装 xvfb。")
        return None
    display_id = os.environ.get("NOTEBOOK_XVFB_DISPLAY", ":99")
    _XVFB_PROCESS = subprocess.Popen(
        [xvfb_bin, display_id, "-screen", "0", "1280x1024x24"],
        stdout=subprocess.DEVNULL,
        stderr=subprocess.DEVNULL,
    )
    os.environ["DISPLAY"] = display_id
    print("已启动 Notebook 内部 Xvfb：DISPLAY =", display_id)
    return _XVFB_PROCESS


def ensure_project_layout():
    required = [PROJECT_ROOT / "asset" / "example_scene_y2.xml", PROJECT_ROOT / "mujoco_env"]
    missing = [path for path in required if not path.exists()]
    if missing:
        print("当前 PROJECT_ROOT 还不是可运行工程，缺少：")
        for path in missing:
            print(" -", public_path(path))
        print("请先设置 PROJECT_ROOT，再运行训练或评估单元。")
        return False
    return True


def write_json_yaml(path, payload):
    path = Path(path)
    path.parent.mkdir(parents=True, exist_ok=True)
    try:
        import yaml
        text = yaml.safe_dump(payload, allow_unicode=True, sort_keys=False)
    except Exception:
        text = json.dumps(payload, ensure_ascii=False, indent=2) + "\n"
    path.write_text(text, encoding="utf-8")
    print("写出配置：", public_path(path))
    return path


def make_lerobot_train_config(policy_type, dataset_repo_id, dataset_root, output_dir, steps, batch_size, chunk_size, n_action_steps, seed=42):
    save_freq = int(os.environ.get(f"{policy_type.upper()}_SAVE_FREQ", os.environ.get("SAVE_FREQ", str(steps))))
    return {
        "dataset": {
            "repo_id": dataset_repo_id,
            "root": str(dataset_root),
            "use_imagenet_stats": True,
        },
        "policy": {
            "type": policy_type,
            "chunk_size": int(chunk_size),
            "n_action_steps": int(n_action_steps),
            "device": "cuda",
        },
        "output_dir": str(output_dir),
        "job_name": Path(output_dir).name,
        "batch_size": int(batch_size),
        "steps": int(steps),
        "save_freq": max(1, save_freq),
        "log_freq": 20,
        "num_workers": 4,
        "seed": int(seed),
        "resume": False,
        "eval_freq": -1,
        "save_checkpoint": True,
        "use_policy_training_preset": True,
        "wandb": {"enable": False, "disable_artifact": True},
    }




class _NotebookCompactProgress:
    def __init__(self, iterable, desc, total=None):
        self.iterable = iterable
        self.desc = desc
        self.total = int(total if total is not None else len(iterable))
        self.every = max(1, int(os.environ.get("NOTEBOOK_PROGRESS_EVERY", "1")))
        self.postfix = ""

    def set_postfix(self, **kwargs):
        self.postfix = ", ".join(f"{key}={value}" for key, value in kwargs.items())

    def __iter__(self):
        for index, value in enumerate(self.iterable, start=1):
            yield value
            if index == 1 or index == self.total or index % self.every == 0:
                suffix = f" | {self.postfix}" if self.postfix else ""
                print(f"{self.desc}: {index}/{self.total}{suffix}", flush=True)

def notebook_progress(iterable, desc, total=None):
    return _NotebookCompactProgress(iterable, desc, total=total)


def train_lerobot_config_in_notebook(config_path, enabled=False, progress_name="train"):
    """Run LeRobot offline training directly inside the notebook kernel.

    The notebook cell owns dataset creation, policy creation, optimizer steps,
    checkpoint saving, tqdm progress, and metric JSONL writing.
    """
    config_path = Path(config_path)
    print("config =", public_path(config_path))
    if not enabled:
        print("未启动。设置 RUN_SMOKE=1 或 RUN_LONG_TRAIN=1 后，本单元会直接在 Notebook 内训练。")
        return None
    if not ensure_project_layout():
        return None

    import time
    from datetime import datetime, timezone
    from contextlib import nullcontext

    import draccus
    import torch
    from torch.amp import GradScaler

    from lerobot.common.datasets.factory import make_dataset
    from lerobot.common.datasets.sampler import EpisodeAwareSampler
    from lerobot.common.datasets.utils import cycle
    from lerobot.common.optim.factory import make_optimizer_and_scheduler
    from lerobot.common.policies.factory import make_policy
    from lerobot.common.policies.utils import get_device_from_parameters
    from lerobot.common.utils.random_utils import set_seed
    from lerobot.common.utils.train_utils import get_step_checkpoint_dir, save_checkpoint, update_last_checkpoint
    from lerobot.common.utils.utils import get_safe_torch_device
    from lerobot.configs.train import TrainPipelineConfig

    class _CompactProgress:
        def __init__(self, iterable, desc, total=None):
            self.iterable = iterable
            self.desc = desc
            self.total = int(total if total is not None else len(iterable))
            self.every = max(1, int(os.environ.get("NOTEBOOK_PROGRESS_EVERY", "100")))
            self.postfix = ""

        def set_postfix(self, **kwargs):
            self.postfix = ", ".join(f"{key}={value}" for key, value in kwargs.items())

        def __iter__(self):
            for index, value in enumerate(self.iterable, start=1):
                yield value
                if index == 1 or index == self.total or index % self.every == 0:
                    suffix = f" | {self.postfix}" if self.postfix else ""
                    print(f"{self.desc}: {index}/{self.total}{suffix}", flush=True)

    def notebook_progress(iterable, desc, total=None):
        return _CompactProgress(iterable, desc, total=total)

    run_started_at = datetime.now(timezone.utc).isoformat()
    cfg = draccus.parse(TrainPipelineConfig, config_path=config_path, args=[])
    cfg.validate()
    if cfg.seed is not None:
        set_seed(cfg.seed)

    device = get_safe_torch_device(cfg.policy.device, log=True)
    torch.backends.cudnn.benchmark = True
    torch.backends.cuda.matmul.allow_tf32 = True

    print("Creating dataset...")
    dataset = make_dataset(cfg)
    print("Creating policy...")
    pretrained_override = os.environ.get(f"{cfg.policy.type.upper()}_PRETRAINED_PATH_OVERRIDE") or os.environ.get("POLICY_PRETRAINED_PATH_OVERRIDE")
    if pretrained_override and not cfg.resume:
        cfg.policy.pretrained_path = str(Path(pretrained_override))
        print("pretrained override =", public_path(cfg.policy.pretrained_path))
    elif cfg.policy.type == "pi0" and not cfg.resume:
        cfg.policy.pretrained_path = "lerobot/pi0"
    elif cfg.policy.type == "smolvla" and not cfg.resume:
        smolvla_base_candidates = [
            os.environ.get("SMOLVLA_BASE_PATH"),
            os.environ.get("SMOLVLA_PRETRAINED_BASE_PATH"),
            str(MODEL_ROOT / "smolvla_base" / "pretrained_model"),
            str(MODEL_ROOT / "lerobot_smolvla_base_legacy"),
            str(MODEL_ROOT / "lerobot_smolvla_base"),
        ]
        local_smolvla_base = next((Path(p) for p in smolvla_base_candidates if p and Path(p).exists()), None)
        if local_smolvla_base is not None:
            cfg.policy.pretrained_path = str(local_smolvla_base)
            print("local smolvla base =", public_path(cfg.policy.pretrained_path))
        else:
            cfg.policy.pretrained_path = "lerobot/smolvla_base"
    if cfg.policy.type == "smolvla":
        local_vlm_model = os.environ.get("SMOLVLA_VLM_MODEL_PATH")
        if local_vlm_model and Path(local_vlm_model).exists():
            cfg.policy.vlm_model_name = str(Path(local_vlm_model))
            cfg.policy.load_vlm_weights = False
            print("local smolvlm processor/config =", public_path(cfg.policy.vlm_model_name))

    policy = make_policy(cfg=cfg.policy, ds_meta=dataset.meta)

    # Compatibility for newer Transformers: PaliGemmaForConditionalGeneration may expose
    # language_model as GemmaModel directly, while this LeRobot Pi0 code expects
    # language_model.model.  Use a non-Module proxy so checkpoints/state_dict stay clean.
    if cfg.policy.type == "pi0":
        try:
            lm = policy.model.paligemma_with_expert.paligemma.language_model
            if not hasattr(lm, "model"):
                class _LanguageModelCoreProxy:
                    def __init__(self, core):
                        self._core = core

                    def __getattr__(self, name):
                        return getattr(self._core, name)

                object.__setattr__(lm, "model", _LanguageModelCoreProxy(lm))
                print("patched Pi0 PaliGemma language_model.model compatibility proxy")
        except Exception as exc:
            print(f"Pi0 PaliGemma compatibility patch skipped: {exc}")

    policy.to(device)
    policy.train()

    optimizer, lr_scheduler = make_optimizer_and_scheduler(cfg, policy)
    grad_scaler = GradScaler(device.type, enabled=cfg.policy.use_amp)

    def _dataset_column_values(name):
        hf_dataset = getattr(dataset, "hf_dataset", None)
        if hf_dataset is None or name not in getattr(hf_dataset, "column_names", []):
            return None
        values = hf_dataset[name]
        try:
            return list(values)
        except TypeError:
            return [values[i] for i in range(len(values))]

    def _task_name_map():
        meta = getattr(dataset, "meta", None)
        tasks = getattr(meta, "tasks", None)
        if tasks is None:
            return {}
        if isinstance(tasks, dict):
            return {int(k): str(v) for k, v in tasks.items()}
        try:
            return {int(k): str(v) for k, v in dict(tasks).items()}
        except Exception:
            return {}

    def _make_weighted_sampler(generator):
        mode = os.environ.get("NOTEBOOK_FRAME_WEIGHT_MODE", "").strip().lower()
        if not mode or mode in {"0", "none", "off", "false"}:
            return None, {"mode": "none"}
        weights = torch.ones(len(dataset), dtype=torch.double)
        info = {"mode": mode, "num_frames": len(dataset)}

        if "blue" in mode:
            blue_weight = float(os.environ.get("NOTEBOOK_BLUE_WEIGHT", "2.0"))
            mask = [False] * len(dataset)
            task_indices = _dataset_column_values("task_index")
            task_names = _task_name_map()
            if task_indices is not None and task_names:
                for idx, task_index in enumerate(task_indices):
                    task_text = task_names.get(int(task_index), "").lower()
                    mask[idx] = ("blue" in task_text) or ("蓝" in task_text)
            else:
                for column in ["task", "language_instruction", "instruction"]:
                    values = _dataset_column_values(column)
                    if values is None:
                        continue
                    for idx, value in enumerate(values):
                        text = str(value).lower()
                        mask[idx] = ("blue" in text) or ("蓝" in text)
                    break
            blue_count = int(sum(mask))
            if blue_count == 0:
                print("警告：NOTEBOOK_FRAME_WEIGHT_MODE=blue 但没有识别到 blue/蓝 指令帧，采样退回均匀。")
            else:
                for idx, is_blue in enumerate(mask):
                    if is_blue:
                        weights[idx] *= blue_weight
            info.update({"blue_weight": blue_weight, "blue_frames": blue_count})

        weight_file = os.environ.get("NOTEBOOK_FRAME_WEIGHT_JSON")
        if weight_file:
            payload = json.loads(Path(weight_file).read_text(encoding="utf-8"))
            for key, value in payload.items():
                weights[int(key)] *= float(value)
            info.update({"weight_json": public_path(weight_file), "json_entries": len(payload)})

        if float(weights.sum()) <= 0:
            raise ValueError("采样权重总和为 0。")
        sampler = torch.utils.data.WeightedRandomSampler(
            weights=weights,
            num_samples=len(weights),
            replacement=True,
            generator=generator,
        )
        info.update(
            {
                "weight_min": float(weights.min()),
                "weight_max": float(weights.max()),
                "weight_mean": float(weights.mean()),
            }
        )
        return sampler, info

    generator = torch.Generator()
    if cfg.seed is not None:
        generator.manual_seed(int(cfg.seed))

    weighted_sampler, sampler_info = _make_weighted_sampler(generator)
    if weighted_sampler is not None:
        shuffle = False
        sampler = weighted_sampler
        print("Notebook weighted sampler =", json.dumps(sampler_info, ensure_ascii=False))
    elif hasattr(cfg.policy, "drop_n_last_frames"):
        shuffle = False
        sampler = EpisodeAwareSampler(
            dataset.episode_data_index,
            drop_n_last_frames=cfg.policy.drop_n_last_frames,
            shuffle=True,
        )
    else:
        shuffle = True
        sampler = None

    dataloader = torch.utils.data.DataLoader(
        dataset,
        num_workers=cfg.num_workers,
        batch_size=cfg.batch_size,
        shuffle=shuffle,
        sampler=sampler,
        generator=generator if sampler is None else None,
        pin_memory=device.type != "cpu",
        drop_last=False,
    )
    dl_iter = cycle(dataloader)

    output_dir = Path(cfg.output_dir)
    output_dir.mkdir(parents=True, exist_ok=True)
    metrics_path = output_dir / "notebook_train_metrics.jsonl"
    num_learnable = sum(p.numel() for p in policy.parameters() if p.requires_grad)
    num_total = sum(p.numel() for p in policy.parameters())
    print(f"output_dir = {public_path(output_dir)}")
    print(f"steps = {cfg.steps}, batch_size = {cfg.batch_size}, frames = {dataset.num_frames}, episodes = {dataset.num_episodes}")
    print(f"learnable_params = {num_learnable:,}, total_params = {num_total:,}")

    last_metrics = None
    progress = notebook_progress(range(1, cfg.steps + 1), desc=progress_name, total=cfg.steps)
    start_all = time.perf_counter()
    for step in progress:
        load_start = time.perf_counter()
        batch = next(dl_iter)
        data_s = time.perf_counter() - load_start
        for key, value in batch.items():
            if isinstance(value, torch.Tensor):
                batch[key] = value.to(device, non_blocking=True)

        update_start = time.perf_counter()
        device_from_policy = get_device_from_parameters(policy)
        with torch.autocast(device_type=device_from_policy.type) if cfg.policy.use_amp else nullcontext():
            loss, output_dict = policy.forward(batch)
        grad_scaler.scale(loss).backward()
        grad_scaler.unscale_(optimizer)
        grad_norm = torch.nn.utils.clip_grad_norm_(
            policy.parameters(),
            cfg.optimizer.grad_clip_norm,
            error_if_nonfinite=False,
        )
        grad_scaler.step(optimizer)
        grad_scaler.update()
        optimizer.zero_grad()
        if lr_scheduler is not None:
            lr_scheduler.step()
        if hasattr(policy, "update"):
            policy.update()
        update_s = time.perf_counter() - update_start

        is_log_step = cfg.log_freq > 0 and (step % cfg.log_freq == 0 or step == 1 or step == cfg.steps)
        is_saving_step = cfg.save_checkpoint and (step % cfg.save_freq == 0 or step == cfg.steps)
        if is_log_step:
            last_metrics = {
                "step": step,
                "loss": float(loss.detach().cpu()),
                "grad_norm": float(grad_norm.detach().cpu()) if hasattr(grad_norm, "detach") else float(grad_norm),
                "lr": float(optimizer.param_groups[0]["lr"]),
                "update_s": float(update_s),
                "data_s": float(data_s),
                "elapsed_s": float(time.perf_counter() - start_all),
            }
            with metrics_path.open("a", encoding="utf-8") as f:
                f.write(json.dumps(last_metrics, ensure_ascii=False) + "\n")
            progress.set_postfix(
                loss=f"{last_metrics['loss']:.4f}",
                lr=f"{last_metrics['lr']:.1e}",
                updt_s=f"{last_metrics['update_s']:.3f}",
            )
        if is_saving_step:
            checkpoint_dir = get_step_checkpoint_dir(cfg.output_dir, cfg.steps, step)
            print(f"\nSaving checkpoint at step {step}: {public_path(checkpoint_dir)}")
            save_checkpoint(checkpoint_dir, step, cfg, policy, optimizer, lr_scheduler)
            update_last_checkpoint(checkpoint_dir)

    run_finished_at = datetime.now(timezone.utc).isoformat()
    duration_s = float(time.perf_counter() - start_all)
    checkpoint_paths = sorted(str(path) for path in output_dir.glob("checkpoints/*/pretrained_model"))
    run_summary = {
        "recipe": str(cfg.policy.type),
        "native_notebook_kernel": True,
        "started_at_utc": run_started_at,
        "finished_at_utc": run_finished_at,
        "duration_s": duration_s,
        "steps": int(cfg.steps),
        "batch_size": int(cfg.batch_size),
        "output_dir": str(output_dir),
        "metrics_path": str(metrics_path),
        "checkpoint_paths": checkpoint_paths,
        "last_metrics": last_metrics,
    }
    summary_path = output_dir / "training_run_summary.json"
    summary_path.write_text(json.dumps(run_summary, ensure_ascii=False, indent=2), encoding="utf-8")
    print("训练完成。metrics =", public_path(metrics_path))
    print("训练耗时 =", f"{duration_s / 60.0:.2f} 分钟")
    print("训练摘要 =", public_path(summary_path))
    if last_metrics is not None:
        print(json.dumps(last_metrics, ensure_ascii=False, indent=2))
    return {"output_dir": output_dir, "metrics_path": metrics_path, "summary_path": summary_path, "last_metrics": last_metrics, "duration_s": duration_s}


def load_eval_module():
    import importlib.util

    if not EVAL_SCRIPT.exists():
        raise FileNotFoundError(f"评估脚本不存在：{public_path(EVAL_SCRIPT)}")
    spec = importlib.util.spec_from_file_location("notebook_eval_policy_success", EVAL_SCRIPT)
    module = importlib.util.module_from_spec(spec)
    assert spec.loader is not None
    spec.loader.exec_module(module)
    return module


def run_eval_policy_in_notebook(
    policy_name,
    policy_path,
    result_path,
    episodes,
    seed_start,
    render=False,
    enabled=False,
    repo_id=None,
    dataset_root=None,
    video_dir=None,
    trace_dir=None,
    seed_list=None,
):
    print("policy =", policy_name)
    print("policy_path =", public_path(policy_path))
    print("result =", public_path(result_path))
    if not enabled:
        print("未启动。设置 RUN_EVAL=1 后，本单元会在 Notebook 内直接加载策略并闭环评估。")
        return None
    if not ensure_project_layout():
        return None

    import argparse
    from contextlib import contextmanager

    @contextmanager
    def pushd(path):
        old = Path.cwd()
        os.chdir(path)
        try:
            yield
        finally:
            os.chdir(old)

    ensure_xvfb_display()
    module = load_eval_module()
    result_path = Path(result_path)
    result_path.parent.mkdir(parents=True, exist_ok=True)
    if result_path.exists():
        result_path.unlink()

    args = argparse.Namespace(
        policy=policy_name,
        episodes=int(episodes),
        seed_start=int(seed_start),
        max_action_steps=int(os.environ.get("EVAL_MAX_ACTION_STEPS", "400")),
        hz=float(os.environ.get("EVAL_HZ", "20")),
        render=bool(render),
        output_jsonl=result_path,
        video_dir=Path(video_dir) if video_dir else None,
        trace_dir=Path(trace_dir) if trace_dir else None,
        device=os.environ.get("EVAL_DEVICE", "cuda"),
        reset_policy_each_action=env_flag("EVAL_RESET_POLICY_EACH_ACTION", False),
        act_n_action_steps=None,
        act_force_dataset_gripper=False,
        act_clamp_timestamp=False,
        act_policy_path=Path(policy_path),
        act_repo_id=repo_id or "datawhale_eai_pnp",
        act_dataset_root=Path(dataset_root or "./demo_data"),
        act_episode_timestamp_offsets="",
        act_episode_source_flags="",
        physical_success=env_flag("EVAL_PHYSICAL_SUCCESS", True),
        physical_min_lift=float(os.environ.get("EVAL_PHYSICAL_MIN_LIFT", "0.06")),
        physical_min_lift_steps=int(os.environ.get("EVAL_PHYSICAL_MIN_LIFT_STEPS", "3")),
        physical_final_upright_cos=float(os.environ.get("EVAL_PHYSICAL_FINAL_UPRIGHT_COS", "0.85")),
        smolvla_policy_path=Path(policy_path),
        pi0_policy_path=Path(policy_path),
        pi0_repo_id=repo_id or os.environ.get("PI0_DATASET_REPO_ID", "datawhale_eai_pnp_language"),
        pi0_dataset_root=Path(dataset_root or os.environ.get("PI0_DATASET_ROOT", "./demo_data_language")),
    )

    with pushd(PROJECT_ROOT):
        selected_cases = set()
        selected_cases = set()
        selected_cases = set()
        if policy_name == "act":
            policy = module.make_act_policy(
                args.device,
                args.act_policy_path,
                args.act_repo_id,
                args.act_dataset_root,
                n_action_steps=args.act_n_action_steps,
                episode_timestamp_offsets=args.act_episode_timestamp_offsets,
                episode_source_flags=args.act_episode_source_flags,
            )
            rollout = module.rollout_act
        elif policy_name == "smolvla":
            policy = module.make_smolvla_policy(args.device, args.smolvla_policy_path)
            rollout = module.rollout_language_policy
        elif policy_name == "pi0":
            policy = module.make_pi0_policy(args.device, args.pi0_policy_path, args.pi0_repo_id, args.pi0_dataset_root)
            rollout = module.rollout_language_policy
        else:
            raise ValueError(policy_name)

        rows = []
        for offset in notebook_progress(range(args.episodes), desc=f"{policy_name} eval", total=args.episodes):
            seed = (list(seed_list)[offset] if seed_list is not None else args.seed_start + offset)
            row = rollout(args, policy, seed)
            row = module.finalize_case_media(args, row, selected_cases)
            rows.append(row)
            with result_path.open("a", encoding="utf-8") as f:
                f.write(json.dumps(row, ensure_ascii=False) + "\n")
            print(json.dumps(row, ensure_ascii=False))
    summarize_jsonl(result_path)
    return rows


def list_checkpoints(run_dir):
    run_dir = Path(run_dir)
    candidates = []
    for pattern in ["checkpoints/*/pretrained_model", "checkpoint*/pretrained_model", "*/pretrained_model", "pretrained_model"]:
        candidates.extend(run_dir.glob(pattern))
    unique = sorted(set(candidates))
    if not unique:
        print("尚未发现 checkpoint：", public_path(run_dir))
        return []
    for path in unique:
        print(" -", public_path(path))
    return unique


def require_trained_policy(run_dir, label="本轮训练"):
    checkpoints = list_checkpoints(run_dir)
    if not checkpoints:
        raise RuntimeError(
            f"{label}没有发现 checkpoint：{public_path(run_dir)}。"
            "请先在本 Notebook 执行训练单元格，不能回退到历史或预训练权重。"
        )
    path = checkpoints[-1]
    if not path.exists():
        raise FileNotFoundError(f"训练 checkpoint 不存在：{public_path(path)}")
    print(f"评估绑定到{label} checkpoint：", public_path(path))
    return path


def summarize_jsonl(path):
    path = Path(path)
    if not path.exists():
        print("结果 JSONL 尚不存在：", public_path(path))
        return None
    rows = [json.loads(line) for line in path.read_text(encoding="utf-8").splitlines() if line.strip()]
    total = len(rows)
    legacy = sum(bool(row.get("success") or row.get("legacy_success")) for row in rows)
    if rows and all("physical_success" in row for row in rows):
        physical_count = sum(bool(row.get("physical_success")) for row in rows)
        physical_text = str(physical_count) + "/" + str(total)
    else:
        physical_text = "未记录"
    md_table(
        ["结果文件", "episodes", "legacy_success", "physical_success"],
        [(public_path(path), total, f"{legacy}/{total}", physical_text)],
    )
    return rows


## 1. 运行控制

本 Notebook 默认执行保护训练和评估。首次运行可先将 `RUN_PROTECTED_TRAIN=False` 做环境检查；正式复现时保持 `True`。训练进度每 100 步刷新一次。


In [ ]:
# RUN_CONTROL_CELL
RUN_SMOKE = False
RUN_LONG_TRAIN = False
RUN_EVAL = True
RUN_PROTECTED_TRAIN = True
os.environ["NOTEBOOK_PROGRESS_EVERY"] = os.environ.get("NOTEBOOK_PROGRESS_EVERY", "100")
print({"RUN_SMOKE": RUN_SMOKE, "RUN_LONG_TRAIN": RUN_LONG_TRAIN, "RUN_EVAL": RUN_EVAL, "RUN_PROTECTED_TRAIN": RUN_PROTECTED_TRAIN})


## 2. 保护训练配方

保护初始化权重只用于训练初始化；最终评估只能使用本单元新生成的最终 checkpoint。


In [ ]:
DATASET_REPO_ID = os.environ.get("DATASET_REPO_ID", "datawhale_eai_pnp_language")
TRAIN_DATA_ROOT = Path(os.environ.get("TRAIN_DATA_ROOT", DATA_ROOT / "demo_data_language"))
CONFIG_DIR = OUTPUT_ROOT / "configs"
print("DATASET_REPO_ID =", DATASET_REPO_ID)
print("TRAIN_DATA_ROOT =", public_path(TRAIN_DATA_ROOT))


### 3. 执行 protected parent + weighted-blue 训练

SmolVLA 先训练 parent 5000 steps，再训练 blue 加权续训 1000 steps。两阶段都会写入自己的训练时间摘要。


In [ ]:
# PROTECTED_TRAIN_CELL
protected_train_enabled = globals().get("RUN_PROTECTED_TRAIN", env_flag("RUN_PROTECTED_TRAIN", False))
if not protected_train_enabled:
    print("未启动。设置 RUN_PROTECTED_TRAIN=1 后，本单元会原生训练 SmolVLA protected recipe。")
else:
    DATASET_REPO_ID = globals().get("DATASET_REPO_ID", os.environ.get("DATASET_REPO_ID", "datawhale_eai_pnp_language"))
    TRAIN_DATA_ROOT = globals().get("TRAIN_DATA_ROOT", Path(os.environ.get("TRAIN_DATA_ROOT", DATA_ROOT / "demo_data_language")))
    CONFIG_DIR = OUTPUT_ROOT / "configs"
    RUN_ROOT = OUTPUT_ROOT / "runs" / "smolvla_protected_recipe"
    PARENT_OUTPUT = RUN_ROOT / "parent_5000"
    WEIGHTED_OUTPUT = RUN_ROOT / "weighted_blue2_step1000"
    parent_config = make_lerobot_train_config(
        "smolvla", DATASET_REPO_ID, TRAIN_DATA_ROOT, PARENT_OUTPUT,
        steps=int(os.environ.get("SMOLVLA_PARENT_STEPS", "5000")),
        batch_size=int(os.environ.get("SMOLVLA_BATCH_SIZE", "4")),
        chunk_size=50,
        n_action_steps=50,
    )
    weighted_config = make_lerobot_train_config(
        "smolvla", DATASET_REPO_ID, TRAIN_DATA_ROOT, WEIGHTED_OUTPUT,
        steps=int(os.environ.get("SMOLVLA_WEIGHTED_STEPS", "1000")),
        batch_size=int(os.environ.get("SMOLVLA_BATCH_SIZE", "4")),
        chunk_size=50,
        n_action_steps=50,
    )
    parent_path = write_json_yaml(CONFIG_DIR / "smolvla_protected_parent_5000.yaml", parent_config)
    weighted_path = write_json_yaml(CONFIG_DIR / "smolvla_protected_weighted_blue2.yaml", weighted_config)
    train_lerobot_config_in_notebook(parent_path, enabled=True, progress_name="SmolVLA protected parent")
    parent_ckpt = list_checkpoints(PARENT_OUTPUT)[-1]
    old_override = os.environ.get("SMOLVLA_PRETRAINED_PATH_OVERRIDE")
    old_mode = os.environ.get("NOTEBOOK_FRAME_WEIGHT_MODE")
    old_blue = os.environ.get("NOTEBOOK_BLUE_WEIGHT")
    os.environ["SMOLVLA_PRETRAINED_PATH_OVERRIDE"] = str(parent_ckpt)
    os.environ["NOTEBOOK_FRAME_WEIGHT_MODE"] = "blue"
    os.environ["NOTEBOOK_BLUE_WEIGHT"] = os.environ.get("SMOLVLA_BLUE_WEIGHT", "2.0")
    try:
        train_lerobot_config_in_notebook(weighted_path, enabled=True, progress_name="SmolVLA protected weighted-blue")
    finally:
        if old_override is None:
            os.environ.pop("SMOLVLA_PRETRAINED_PATH_OVERRIDE", None)
        else:
            os.environ["SMOLVLA_PRETRAINED_PATH_OVERRIDE"] = old_override
        if old_mode is None:
            os.environ.pop("NOTEBOOK_FRAME_WEIGHT_MODE", None)
        else:
            os.environ["NOTEBOOK_FRAME_WEIGHT_MODE"] = old_mode
        if old_blue is None:
            os.environ.pop("NOTEBOOK_BLUE_WEIGHT", None)
        else:
            os.environ["NOTEBOOK_BLUE_WEIGHT"] = old_blue
    print("protected candidate checkpoints:")
    protected_candidates = list_checkpoints(WEIGHTED_OUTPUT)
    if not protected_candidates:
        raise RuntimeError("SmolVLA protected 续训结束后没有 checkpoint，禁止进入评估。")
    TRAINED_POLICY_PATH = protected_candidates[-1]
    TRAINING_SUMMARY_PATHS = [
        PARENT_OUTPUT / "training_run_summary.json",
        WEIGHTED_OUTPUT / "training_run_summary.json",
    ]
    print("本 Notebook protected checkpoint =", public_path(TRAINED_POLICY_PATH))


## 4. 训练耗时与 checkpoint

下面只读取本次训练写入的 `training_run_summary.json`，显示开始时间、结束时间、总耗时、步数和输出目录。


In [ ]:
summary_paths = [Path(path) for path in globals().get("TRAINING_SUMMARY_PATHS", [])]
if not summary_paths:
    print("尚未执行 protected 训练，暂无训练耗时。")
else:
    summaries = []
    for summary_path in summary_paths:
        if summary_path.exists():
            summaries.append(json.loads(summary_path.read_text(encoding="utf-8")))
        else:
            print("尚未生成：", public_path(summary_path))
    if summaries:
        md_table(
            ["recipe", "steps", "duration_min", "started_at_utc", "finished_at_utc", "output_dir"],
            [(item.get("recipe"), item.get("steps"), f"{float(item.get('duration_s', 0.0))/60.0:.2f}", item.get("started_at_utc"), item.get("finished_at_utc"), item.get("output_dir")) for item in summaries],
        )


## 5. 14 条闭环评估

固定 seed 逐条运行，保存结果 JSONL、成功/失败视频和动作 trace。


In [ ]:
# FULL_14_EVAL_WITH_MEDIA
FULL_EVAL_EPISODES = 14
FULL_EVAL_SEEDS = list(range(14))
FULL_EVAL_ROOT = OUTPUT_ROOT / "full_eval_14"
FULL_EVAL_RESULT = FULL_EVAL_ROOT / "result.jsonl"
if not globals().get("RUN_EVAL", False):
    print("未启动评估。请将 RUN_EVAL=True 后执行本单元。")
else:
    if globals().get("TRAINED_POLICY_PATH") is None:
        raise RuntimeError("本 Notebook 没有当前训练 checkpoint；请先执行本 Notebook 的训练单元格。")
    FULL_EVAL_POLICY = Path(TRAINED_POLICY_PATH)
    if not FULL_EVAL_POLICY.exists():
        raise FileNotFoundError(f"当前训练 checkpoint 不存在：{public_path(FULL_EVAL_POLICY)}")
    print("评估模型（来自本 Notebook 当前训练）：", public_path(FULL_EVAL_POLICY))
    FULL_EVAL_ROWS = run_eval_policy_in_notebook(
        "smolvla", FULL_EVAL_POLICY, FULL_EVAL_RESULT,
        episodes=FULL_EVAL_EPISODES, seed_start=0, seed_list=FULL_EVAL_SEEDS,
        render=True, enabled=True, repo_id=DATASET_REPO_ID, dataset_root=TRAIN_DATA_ROOT,
        video_dir=FULL_EVAL_ROOT / "videos", trace_dir=FULL_EVAL_ROOT / "traces",
    )
FULL_EVAL_ROWS = run_eval_policy_in_notebook(
    "smolvla",
    FULL_EVAL_POLICY,
    FULL_EVAL_RESULT,
    episodes=FULL_EVAL_EPISODES,
    seed_start=0,
    seed_list=FULL_EVAL_SEEDS,
    render=True,
    enabled=RUN_EVAL,
    repo_id=DATASET_REPO_ID,
    dataset_root=TRAIN_DATA_ROOT,
    video_dir=FULL_EVAL_ROOT / "videos",
    trace_dir=FULL_EVAL_ROOT / "traces",
)


In [ ]:
# FULL_14_MEDIA_REVIEW
from IPython.display import Video
import matplotlib.pyplot as plt

def display_full_eval_media(result_path, title="smolvla - 14 episode evaluation"):
    result_path = Path(result_path)
    if not result_path.exists():
        print("Evaluation has not run yet:", public_path(result_path))
        return
    rows = [json.loads(line) for line in result_path.read_text(encoding="utf-8").splitlines() if line.strip()]
    success = sum(bool(row.get("physical_success", row.get("success"))) for row in rows)
    print(f"{title}: physical_success={success}/{len(rows)}")
    md_table([["metric", "value"]][0], [["physical success", f"{success}/{len(rows)}"]])
    for kind in ("success", "failure"):
        row = next((item for item in rows if bool(item.get("physical_success", item.get("success"))) == (kind == "success") and item.get("video_path")), None)
        if row is None:
            print(f"No {kind} episode was found.")
            continue
        print(f"{kind} case: seed={row['seed']}")
        video_path = Path(row["video_path"])
        if video_path.exists():
            display(Video(filename=str(video_path), embed=False, width=960))
        trace_path = Path(row.get("trace_path", ""))
        if not trace_path.exists():
            continue
        trace = json.loads(trace_path.read_text(encoding="utf-8"))
        actions = np.asarray([item.get("action", []) for item in trace], dtype=np.float32)
        if actions.ndim != 2 or not len(actions):
            continue
        fig, ax = plt.subplots(figsize=(12, 4))
        channels = min(actions.shape[1], 7)
        for channel in range(channels):
            label = "gripper" if channel == 6 else f"action[{channel}]"
            ax.plot(actions[:, channel], label=label, linewidth=1.2)
        ax.set_title(f"{title} - {kind} sequence, seed={row['seed']}")
        ax.set_xlabel("policy step")
        ax.set_ylabel("normalized action / command")
        ax.grid(alpha=0.25)
        ax.legend(ncol=4, fontsize=8)
        plt.tight_layout()
        display(fig)
        plt.close(fig)

display_full_eval_media(FULL_EVAL_RESULT)


## 6. 复现检查

评估结果必须来自本 Notebook 当前训练生成的 `TRAINED_POLICY_PATH`。没有当前 checkpoint 时，评估单元会直接报错。
